<a href="https://colab.research.google.com/github/Mouad-Tazka/-ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# Calculate revenue for each order
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
total_rows = len(df)

print(f"Total Rows: {total_rows}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units}")

Total Rows: 400
Total Revenue: $8,520.00
Total Units Sold: 783



So after setting up the revenue column, we ended up with a total revenue of \$8,520.00 over a total of 783 units sold. This falls right in that expected \$8,000 to $9,000 sweet spot.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [10]:
# Calculate revenue by category
by_category = df.groupby('category')[['revenue']].sum().sort_values(by='revenue', ascending=False)
by_category['percentage'] = (by_category['revenue'] / df['revenue'].sum()) * 100

print(by_category)

          revenue  percentage
category                     
Food       4293.0   50.387324
Merch      1771.5   20.792254
Drink      1554.0   18.239437
RainGear    901.5   10.580986


Food brought in \$4293.00 which makes up 50.4% of our entire sales.
Meanwhile, RainGear is at the very bottom with only $901.50 (10.6%), which makes sense if the weather was nice!

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# Calculate average order revenue and count of orders per vendor
vendor_stats = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values(by='mean', ascending=False)

print(vendor_stats)

# Extract highest and lowest for the student explanation
highest_vendor = vendor_stats.index[0]
highest_avg = vendor_stats.iloc[0]['mean']
highest_count = int(vendor_stats.iloc[0]['count'])

lowest_vendor = vendor_stats.index[-1]
lowest_avg = vendor_stats.iloc[-1]['mean']
lowest_count = int(vendor_stats.iloc[-1]['count'])

                mean  count
vendor_id                  
V-01       22.595745     94
V-18       21.750000    108
V-05       20.580645     93
V-10       20.314286    105


So, looking at the numbers, V-01 has the highest average order revenue at \$22.60 across 94 orders.
On the flip side, V-10 averages the lowest with $20.31 per order, though they did have a pretty solid volume of 105 orders overall!

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# Calculate the percentage share of revenue from Merch
merch_share = by_category.loc['Merch', 'percentage']

print(f"Merch Share of Revenue: {merch_share:.1f}%")

Merch Share of Revenue: 20.8%


Looking at our sales breakdown, Merch brought in exactly 20.8% of our total revenue, basically a fifth of all our sales.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# Perform a left join and validate as many_to_one
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Check if row count and total revenue changed
rows_match = len(joined) == len(df)
revenue_match = abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01

# Find the unmatched vendor id(s)
unmatched_vendors = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()

print(f"Row count unchanged? {rows_match} (Total: {len(joined)} rows)")
print(f"Total revenue unchanged? {revenue_match} (Total: ${joined['revenue'].sum():,.2f})")
print(f"Unmatched vendor ID in orders: {unmatched_vendors}")
print(f"Number of orders affected by unmatched vendor: {len(df[df['vendor_id'] == 'V-18'])}")

Row count unchanged? True (Total: 400 rows)
Total revenue unchanged? True (Total: $8,520.00)
Unmatched vendor ID in orders: ['V-18']
Number of orders affected by unmatched vendor: 108


The total rows stayed at 400 and our revenue didn't budge from \$8,520.00. We found out that vendor V-18 is completely missing from the lookup sheet, leaving 108 orders without a name! I kept them in the table using a left join so we wouldn't lose their sales, though they'll just show up as NaN for now.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [9]:
# Fill NaN values in vendor_name for V-18 to ensure a complete report
joined['vendor_name_clean'] = joined['vendor_name'].fillna('V-18 (Unknown)')

# Create the pivot table with row and column margins
pivot_report = joined.pivot_table(
    values='revenue',
    index='vendor_name_clean',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

print(pivot_report)

category            Drink    Food   Merch  RainGear   Total
vendor_name_clean                                          
Cav Merch North     502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers        171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos       298.5   882.0   489.0     244.5  1914.0
V-18 (Unknown)      582.0  1018.5   508.5     240.0  2349.0
Total              1554.0  4293.0  1771.5     901.5  8520.0


Our highest-selling vendor-category combination is Hoos Burgers' Food sales at \$1,338.00.
V-18 (Unknown) generated the most Merch revenue at \$508.50, while Rotunda Tacos generated the most RainGear revenue at $244.50.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

### Final Write-up

#### **Part a) Recommendations for Next Game**
To maximize overall sales for the upcoming game, vendors should strategically adjust their inventory to reflect actual consumer demand. Food is our primary driver, generating over half of the total revenue with 4,293.00 dollars out of our 8,520.00 dollar total. Because of this massive demand, every vendor should prioritize keeping their food stands heavily stocked. Specifically, Rotunda Tacos underperformed in this category, bringing in only 882.00 dollars in food sales compared to Hoos Burgers at 1,338.00 dollars; they should evaluate their menu or throughput to match Hoos Burgers' success. Additionally, since RainGear contributed just 10.6% to our total revenue ($901.50), vendors should scale back on stocking rain protection items unless a storm is explicitly forecasted.

#### **Part b) Least Trustworthy Answer Analysis**
Question Q3, which determines which vendor has the highest average order revenue, is the least trustworthy calculation in this report. Our lookup directory is completely missing the name and identity of vendor V-18. This missing vendor accounts for 108 of our 400 total orders, representing $2,349.00 in sales. Because more than a quarter of our total transactions belong to an unnamed entity, any ranking or strategic assessment of individual vendor performance remains highly speculative and unvalidated.